In [ ]:
pip install open_clip_torch

In [ ]:
import torch
from PIL import Image
import open_clip
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

### Initialize the model, preprocessing function and tokenizer

In [ ]:
model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-16', pretrained='openai')
model.eval()  # model in train mode by default
model.to(device)
tokenizer = open_clip.get_tokenizer('ViT-B-16')

## Part 1 — Sketch-200: Tip-Adapter across 3 seeds

### Prepare the Sketch-200 Dataset

In [ ]:
pip install datasets==2.16.0

In [ ]:
from datasets import load_dataset
dataset = load_dataset("songweig/imagenet_sketch")

In [ ]:
print(dataset)

In [ ]:
# if sys.modules['imagenet_r_classes']:
#   del sys.modules['imagenet_r_classes']

In [ ]:
from imagenet_r_classes import r_class_names, r_wnids, wnid_to_r_index

In [ ]:
import json
from torchvision.datasets.utils import download_url

# Download the official ImageNet class index mapping
download_url(
    "https://s3.amazonaws.com/deep-learning-models/image-models/imagenet_class_index.json",
    "./",
    "imagenet_class_index.json",
)

# Load the JSON mapping file
with open("./imagenet_class_index.json", "r") as f:
    class_idx = json.load(f)

# Convert to a list where index 0-999 corresponds to model outputs
class_names = [class_idx[str(i)][1] for i in range(1000)]

In [ ]:
wnid_to_idx = {class_idx[str(i)][0]: i for i in range(1000)}
idx_to_wnid = {i: class_idx[str(i)][0] for i in range(1000)}

In [ ]:
r_idx_1000 = [wnid_to_idx[w] for w in r_wnids]
idx_to_r_index = {wnid_to_idx[w]: i for w, i in wnid_to_r_index.items()}

In [ ]:
from datasets import ClassLabel

keep = set(r_idx_1000)
sk200 = dataset.filter(lambda y: y in keep, input_columns="label")

In [ ]:
print(sk200)

In [ ]:
new_features = sk200['train'].features.copy()
new_features["label"] = ClassLabel(names=r_class_names)

sk200 = sk200.map(
    lambda y: {"label": idx_to_r_index[y]},
    input_columns="label",
    features=new_features,
)

In [ ]:
sk200['train'][0]

### Run the harness for difference seeds

In [ ]:
import collections
import random

def split_indices(labels, seed, n_cache=16, n_val=10):
    rng = random.Random(seed)
    by_class = collections.defaultdict(list)
    for i, y in enumerate(labels):
        by_class[y].append(i)
    cache, val, test = [], [], []
    for idx in by_class.values():
        idx = idx[:]
        rng.shuffle(idx)
        cache += idx[:n_cache]
        val += idx[n_cache:n_cache + n_val]
        test += idx[n_cache + n_val:]
    return cache, val, test

In [ ]:
class HFImageDataset(Dataset):
    def __init__(self, hf_dataset, preprocess, wnid_to_index):
        self.hf_dataset = hf_dataset
        self.preprocess = preprocess
        self.wnid_to_index = wnid_to_index

    def __len__(self):
        return len(self.hf_dataset)

    def __getitem__(self, idx):
        example = self.hf_dataset[idx]
        image = self.preprocess(example["image"].convert("RGB"))
        label = example["label"]
        return image, label

In [ ]:
full_wrapped = HFImageDataset(sk200['train'], preprocess, wnid_to_r_index)
full_loader = DataLoader(full_wrapped, batch_size=32)

In [ ]:
sk200_all_features = build_and_cache_image_features(model, device, full_loader, './features', 'sk200_all_features')

### Build the text features for zero shot/

In [ ]:
from clip_zeroshot import load_cached_text_features, build_and_cache_image_features, load_cached_image_features

In [ ]:
text_feature_cache = load_cached_text_features('/content/features/r_text-features.pt')
text_features = text_feature_cache['text_features']

### Run the harness for different seeds

In [ ]:
from harness import run_comparison, zero_shot_logits, tip_adapter_logits, ece, accuracy, signed_gap
import torch.nn.functional as F

metrics = {"accuracy": accuracy, "ece": ece, "signed_gap": signed_gap}
methods = {
    "zero_shot":   {"fn": zero_shot_logits,   "params": {}},
    "tip_adapter": {"fn": tip_adapter_logits, "params": {"alpha": 1.5, "beta": 5.0}}
}


In [ ]:
import pandas as pd

rows = []
labels_all = sk200['train']['label']

for sd in [42, 43, 44]:
    cache_idx, val_idx, test_idx = split_indices(labels_all, seed=sd)

    # split checks, every seed
    assert len(cache_idx) + len(val_idx) + len(test_idx) == len(labels_all)
    assert set(cache_idx).isdisjoint(val_idx)
    assert set(cache_idx).isdisjoint(test_idx)
    assert set(val_idx).isdisjoint(test_idx)

    few_shot_img_feats = sk200_all_features['image_features'][cache_idx]
    few_shot_labels = sk200_all_features['labels'][cache_idx]
    test_feats = sk200_all_features['image_features'][test_idx]
    test_labels = sk200_all_features['labels'][test_idx]

    c = collections.Counter(few_shot_labels.tolist())
    assert min(c.values()) == max(c.values()) == 16

    cache_values = F.one_hot(few_shot_labels, num_classes=len(r_class_names)).float()

    shared = {
        "test_features": test_feats.to(device),
        "labels": test_labels.to(device),
        "text_features": text_features.to(device),
        "cache_keys": few_shot_img_feats.to(device),
        "cache_values": cache_values.to(device),
        "logit_scale": model.logit_scale.exp(),
    }

    res = run_comparison(shared, methods, metrics)
    zs, ta = res["zero_shot"], res["tip_adapter"]

    rows.append({
        "seed": sd,
        "n_test": len(test_idx),
        "zs_gap": zs["signed_gap"],
        "ta_gap": ta["signed_gap"],
        "delta": ta["signed_gap"] - zs["signed_gap"],
        "zs_acc": zs["accuracy"],
        "ta_acc": ta["accuracy"],
        "zs_ece": zs["ece"],
        "ta_ece": ta["ece"],
    })

sk200_draws = pd.DataFrame(rows)
sk200_draws

In [ ]:
summary = sk200_draws.drop(columns=["seed", "n_test"]).agg(["mean", "min", "max"]).T
summary["range"] = summary["max"] - summary["min"]
summary.round(2)

In [ ]:
sk200_draws.to_csv("./features/sk200_seed_draws.csv", index=False)

Sketch-200, α=1.5, β=5.0, 3 seeds. Δ = +7.64 (range 0.93). Seed 42 (Week 1's draw) was the highest. Zero-shot gap also varies by 0.6, from test-set composition alone. Not compared to R yet: R's +6.05 is one draw under the old protocol.

## Part 2 — ImageNet-R: Tip-Adapter across 3 seeds

In [ ]:
r_few = load_cached_image_features('/content/features/r_few_shot_image_features.pt')
r_eval = load_cached_image_features('/content/features/r_eval_features.pt')

r_all_features = {
    "image_features": torch.cat([r_few["image_features"], r_eval["image_features"]]),
    "labels": torch.cat([r_few["labels"], r_eval["labels"]]),
}
torch.save(r_all_features, './features/r_all_features.pt')

In [ ]:
# checks
f, y = r_all_features["image_features"], r_all_features["labels"]
print(f.shape, y.shape)                  # (30000, 512), (30000,)
print(y.min().item(), y.max().item())    # 0, 199
counts = collections.Counter(y.tolist())
print(len(counts), min(counts.values()), max(counts.values()))

In [ ]:
import pandas as pd

rows = []
labels_all = r_all_features["labels"].tolist()

for sd in [42, 43, 44]:
    cache_idx, val_idx, test_idx = split_indices(labels_all, seed=sd)

    # split checks, every seed
    assert len(cache_idx) + len(val_idx) + len(test_idx) == len(labels_all)
    assert set(cache_idx).isdisjoint(val_idx)
    assert set(cache_idx).isdisjoint(test_idx)
    assert set(val_idx).isdisjoint(test_idx)
    print(f"Length of test_idx: {len(test_idx)}")

    few_shot_img_feats = r_all_features['image_features'][cache_idx]
    few_shot_labels = r_all_features['labels'][cache_idx]
    test_feats = r_all_features['image_features'][test_idx]
    test_labels = r_all_features['labels'][test_idx]

    c = collections.Counter(few_shot_labels.tolist())
    assert min(c.values()) == max(c.values()) == 16

    cache_values = F.one_hot(few_shot_labels, num_classes=len(r_class_names)).float()

    shared = {
        "test_features": test_feats.to(device),
        "labels": test_labels.to(device),
        "text_features": text_features.to(device),
        "cache_keys": few_shot_img_feats.to(device),
        "cache_values": cache_values.to(device),
        "logit_scale": model.logit_scale.exp(),
    }

    res = run_comparison(shared, methods, metrics)
    zs, ta = res["zero_shot"], res["tip_adapter"]

    rows.append({
        "seed": sd,
        "n_test": len(test_idx),
        "zs_gap": zs["signed_gap"],
        "ta_gap": ta["signed_gap"],
        "delta": ta["signed_gap"] - zs["signed_gap"],
        "zs_acc": zs["accuracy"],
        "ta_acc": ta["accuracy"],
        "zs_ece": zs["ece"],
        "ta_ece": ta["ece"],
    })

r_seed_draws = pd.DataFrame(rows)
r_seed_draws

In [ ]:
summary = r_seed_draws.drop(columns=["seed", "n_test"]).agg(["mean", "min", "max"]).T
summary["range"] = summary["max"] - summary["min"]
summary.round(2)

In [ ]:
r_seed_draws.to_csv("./features/r_seed_draws.csv", index=False)

#### Result: ImageNet-R, 3 seeds (α=1.5, β=5.0)

Δ = +6.01 on average (range 5.85–6.17). Zero-shot starts at −6.50, close to the old
single-split −6.45, so the concatenation and new split didn't change the baseline.

Compared with Sketch-200 (Part 1):

| | Δ mean | min | max | range |
|---|---|---|---|---|
| ImageNet-R | +6.01 | +5.85 | +6.17 | 0.32 |
| Sketch-200 | +7.64 | +7.25 | +8.17 | 0.93 |

The ranges don't overlap. R's highest seed is still more than a point below Sketch-200's
lowest, so the ~1.6 gap isn't from which images landed in the cache or test set.

Class count and draw noise are now both controlled. Dose isn't: both datasets still use
R's α=1.5, β=5.0. Part 3 retunes it.

Three seeds is a control, not a significance test. CIs come later.

R's spread is smaller because its test set is about 5× larger (24,800 vs 4,952).

R is also imbalanced: 51 to 430 images per class, against 50–51 for Sketch-200. After
taking 16 cache and 10 val per class, the big classes dominate R's test set, and the
metrics are pooled over images, so R's numbers lean toward those classes. R has always
been reported this way, so the comparison stands, but it's a difference between the
two test sets.